# Description

In this notebook, we benchmark EQL with division algorithm on the set of previously generated SR benchmarks.

In [1]:
from __future__ import annotations

import csv
import time
from pathlib import Path

import h5py
import numpy as np
import sympy as sp
import tensorflow as tf

from config.benchmark_config import DataCFG, EQLDIV

from src.EQLdiv.evaluation import (
    calculate_complexity,
    symbolic_matmul_and_bias,
    symbolic_eql_layer,
    get_symbol_list,
)

from src.EQLdiv.utils import get_div_thresh_fn, step_to_epochs
import src.EQLdiv.EQL_Layer_tf as eql
from src.EQLdiv.data_utils import get_penalty_data


# ------------------------------------------------------------
# Data loading (IDENTICAL TO SINDY)
# ------------------------------------------------------------

def _load_one_group(f: h5py.File, gname: str):
    g = f[gname]

    raw = g["sympy_str"][()]
    true_expr_str = raw.decode("utf-8") if isinstance(raw, (bytes, bytearray)) else str(raw)

    Xtr = g["train"]["X"][...].astype(np.float32)
    ytr = g["train"]["y"][...].astype(np.float32).reshape(-1, 1)

    Xti = g["test_interp"]["X"][...].astype(np.float32)
    yti = g["test_interp"]["y"][...].astype(np.float32).reshape(-1)

    Xte = g["test_extrap"]["X"][...].astype(np.float32)
    yte = g["test_extrap"]["y"][...].astype(np.float32).reshape(-1)

    return true_expr_str, Xtr, ytr, Xti, yti, Xte, yte


def _mse(yhat, y):
    yhat = np.asarray(yhat).reshape(-1)
    y = np.asarray(y).reshape(-1)
    return float(np.mean((yhat - y) ** 2))


# ------------------------------------------------------------
# EQL model wrapper
# ------------------------------------------------------------

class EQLDivModel:

    def __init__(self, n_inputs, n_outputs):

        self.metadata = {
            "train_val_examples": 1,
            "num_inputs": n_inputs,
            "num_outputs": n_outputs,
            "extracted_output_bound": 10.0,
        }

        div_thresh_fn = get_div_thresh_fn(
            True,
            EQLDIV.batch_size,
            EQLDIV.test_div_threshold,
            train_examples=1,
        )

        reg_div = lambda repeats: dict(repeats=repeats, div_thresh_fn=div_thresh_fn)

        self.layers = []

        for _ in range(EQLDIV.num_h_layers):
            self.layers.append(
                eql.EQL_Layer(
                    **{op: EQLDIV.layer_width for op in EQLDIV.layer_ops},
                    weight_init_scale=EQLDIV.weight_init_param,
                )
            )

        self.layers.append(
            eql.EQL_Layer(
                reg_div=reg_div(n_outputs),
                weight_init_scale=EQLDIV.weight_init_param,
            )
        )

    def __call__(self, x, global_step):
        out = x
        for layer in self.layers:
            out = layer(out)
        return out


# ------------------------------------------------------------
# Main experiment
# ------------------------------------------------------------

def main():

    cfg = DataCFG()
    out_csv = Path(EQLDIV.results_path)
    out_csv.parent.mkdir(parents=True, exist_ok=True)

    with h5py.File(cfg.h5_path, "r") as f, out_csv.open("w", newline="") as out:

        w = csv.writer(out)
        w.writerow(
            [
                "group",
                "run",
                "seed",
                "train_mse",
                "test_interp_mse",
                "test_extrap_mse",
                "duration_s",
                "true_expr",
                "found_expr",
            ]
        )

        for gname in sorted(f.keys()):

            true_expr_str, Xtr, ytr, Xti, yti, Xte, yte = _load_one_group(f, gname)

            n_inputs = Xtr.shape[1]

            for run in range(EQLDIV.n_runs):

                seed = EQLDIV.base_seed + run
                tf.random.set_seed(seed)
                np.random.seed(seed)

                model = EQLDivModel(n_inputs, 1)
                optimizer = tf.keras.optimizers.Adam(
                    learning_rate=EQLDIV.learning_rate,
                    beta_1=EQLDIV.beta1,
                )

                global_step = tf.Variable(0, dtype=tf.int64)

                t0 = time.perf_counter()

                # training loop
                for _ in range(EQLDIV.epoch_factor):
                    with tf.GradientTape() as tape:
                        preds = model(tf.convert_to_tensor(Xtr), global_step)
                        loss = tf.reduce_mean((preds - ytr) ** 2)

                    grads = tape.gradient(loss, tape.watched_variables())
                    optimizer.apply_gradients(zip(grads, tape.watched_variables()))
                    global_step.assign_add(1)

                dur = time.perf_counter() - t0

                yhat_tr = model(tf.convert_to_tensor(Xtr), global_step).numpy()
                yhat_ti = model(tf.convert_to_tensor(Xti), global_step).numpy()
                yhat_te = model(tf.convert_to_tensor(Xte), global_step).numpy()

                train_mse = _mse(yhat_tr, ytr)
                test_interp_mse = _mse(yhat_ti, yti)
                test_extrap_mse = _mse(yhat_te, yte)

                # symbolic extraction
                kernels = [layer._dense.kernel.numpy() for layer in model.layers]
                biases = [layer._dense.bias.numpy() for layer in model.layers]
                fns_list = [layer.get_fns() for layer in model.layers]

                in_nodes = get_symbol_list(n_inputs)
                res = in_nodes
                for kernel, bias, fns in zip(kernels, biases, fns_list):
                    res = symbolic_matmul_and_bias(res, kernel, bias)
                    res = symbolic_eql_layer(res, fns)

                expr = sp.N(res[0], EQLDIV.round_decimals)
                found_expr_str = str(expr)

                w.writerow(
                    [
                        gname,
                        run,
                        seed,
                        train_mse,
                        test_interp_mse,
                        test_extrap_mse,
                        dur,
                        true_expr_str,
                        found_expr_str,
                    ]
                )
                out.flush()

                print(
                    f"[{gname}] run={run} "
                    f"train={train_mse:.3e} "
                    f"interp={test_interp_mse:.3e} "
                    f"extrap={test_extrap_mse:.3e}"
                )

    print(f"\nSaved: {out_csv}")


if __name__ == "__main__":
    main()

AttributeError: 'dict' object has no attribute 'div_thresh_fn'

In [ ]:
# from __future__ import annotations

# import math
# from pathlib import Path

# import pandas as pd
# import sympy as sp

# from config.benchmark_config import SINDY


# def _safe_latex(expr_str: str) -> str | None:
#     if not isinstance(expr_str, str) or not expr_str.strip():
#         return None
#     try:
#         expr = sp.sympify(expr_str)
#         return sp.latex(expr)
#     except Exception:
#         return None


# def _format_pm(value: float, std: float, sig: int = 1) -> str:
#     if value == 0.0:
#         return r"(0\pm0)\times 10^{0}"

#     exp = int(math.floor(math.log10(abs(value))))
#     scale = 10 ** exp

#     v = round(value / scale, sig)
#     s = round(std / scale, sig)

#     return rf"({v}\pm{s})\times 10^{{{exp}}}"


# def _count_nodes(expr: sp.Expr) -> int:
#     return sum(1 for _ in sp.preorder_traversal(expr))


# def summarize(csv_path: str | Path, k: int = 5) -> None:
#     df = pd.read_csv(csv_path)

#     metrics = [
#         "train_mse",
#         "test_interp_mse",
#         "test_extrap_mse",
#     ]

#     for gname, gdf in df.groupby("group"):
#         print(f"\n{gname}")

#         # Select k best by extrapolation error
#         gdf = gdf.sort_values("test_extrap_mse").iloc[:k]

#         # ---- error metrics ----
#         for m in metrics:
#             vals = gdf[m].astype(float).to_numpy()
#             mean = float(vals.mean())
#             std = float(vals.std(ddof=0))
#             print(f"  {m}: {_format_pm(mean, std)}")

#         # ---- symbolic complexity ----
#         node_counts = []
#         for s in gdf["found_expr"]:
#             if not isinstance(s, str) or not s.strip():
#                 continue
#             try:
#                 expr = sp.sympify(s)
#                 node_counts.append(_count_nodes(expr))
#             except Exception:
#                 pass

#         if node_counts:
#             nc = pd.Series(node_counts, dtype=float)
#             print(
#                 f"  expr_nodes: "
#                 f"{nc.mean():.1f} ± {nc.std(ddof=0):.1f}"
#             )
#         else:
#             print("  expr_nodes: N/A")

#         # ---- best train-fit expression ----
#         best_row = gdf.sort_values("train_mse").iloc[0]
#         latex_expr = _safe_latex(best_row["found_expr"])

#         if latex_expr is not None:
#             print("  best_train_expr_latex:")
#             print(f"    ${latex_expr}$")
#         else:
#             print("  best_train_expr_latex: N/A")


# if __name__ == "__main__":
#     summarize(SINDY.results_path, k=5)


expr_000_lin_uni
  train_mse: (0\pm0)\times 10^{0}
  test_interp_mse: (0\pm0)\times 10^{0}
  test_extrap_mse: (0\pm0)\times 10^{0}
  expr_nodes: 5.0 ± 0.0
  best_train_expr_latex:
    $1.87 x_{1} + 2.01$

expr_001_lin_bi
  train_mse: (5.8\pm0.0)\times 10^{-30}
  test_interp_mse: (5.1\pm0.0)\times 10^{-30}
  test_extrap_mse: (1.5\pm0.0)\times 10^{-29}
  expr_nodes: 8.0 ± 0.0
  best_train_expr_latex:
    $1.56 x_{1} + 1.59 x_{2} - 2.91$

expr_002_poly2_uni
  train_mse: (1.2\pm0.0)\times 10^{-30}
  test_interp_mse: (1.5\pm0.0)\times 10^{-30}
  test_extrap_mse: (6.9\pm0.0)\times 10^{-29}
  expr_nodes: 10.0 ± 0.0
  best_train_expr_latex:
    $2.48 x_{1}^{2} + 1.92 x_{1} - 0.680000000000001$

expr_003_poly2_bi
  train_mse: (5.9\pm0.0)\times 10^{-29}
  test_interp_mse: (5.7\pm0.0)\times 10^{-29}
  test_extrap_mse: (4.9\pm0.0)\times 10^{-28}
  expr_nodes: 22.0 ± 0.0
  best_train_expr_latex:
    $0.550000000000001 x_{1}^{2} + 2.45 x_{1} x_{2} + 1.65 x_{1} + 2.95 x_{2}^{2} + 0.800000000000001 x